# Session 5.3 以降低損失為目標

- Square Error Loss function 對有明顯誤差的預測像有較強的懲罰作用。
- Broadcasting 張量擴張

In [1]:
import torch

t_c = [0.5,  14.0, 15.0, 28.0, 11.0,  8.0,  3.0, -4.0,  6.0, 13.0, 21.0] #攝氏溫度
t_u = [35.7, 55.9, 58.2, 81.9, 56.3, 48.9, 33.9, 21.8, 48.4, 60.4, 68.4] #溫度計上對應到的讀數
t_c = torch.tensor(t_c)
t_u = torch.tensor(t_u)

In [2]:
def model(t_u, w, b):
    return w * t_u +b

def loss_fn(t_p, t_c): #定義損失函數
    squared_diffs = (t_p - t_c)**2
    return squared_diffs.mean()

w = torch.ones(())
b = torch.zeros(())
t_p = model(t_u, w, b)
t_p

tensor([35.7000, 55.9000, 58.2000, 81.9000, 56.3000, 48.9000, 33.9000, 21.8000,
        48.4000, 60.4000, 68.4000])

In [3]:
loss = loss_fn(t_p, t_c)
loss

tensor(1763.8846)

In [4]:
x = torch.ones(()) #0軸
y = torch.ones((3,1)) #2軸 3x1
z = torch.ones((1,3)) #2軸 1x3
a = torch.ones((2,1,1)) #3軸 2x1x1

print(f"shapes: x:{x.shape}, y:{y.shape}")
print(f"shapes: z:{x.shape}, a:{y.shape}")
print("x * y:" ,(x*y).shape)
print("y * z:" ,(y*z).shape)
print("y * z * a:" ,(y*z*a).shape)

shapes: x:torch.Size([]), y:torch.Size([3, 1])
shapes: z:torch.Size([]), a:torch.Size([3, 1])
x * y: torch.Size([3, 1])
y * z: torch.Size([3, 3])
y * z * a: torch.Size([2, 3, 3])


In [7]:
y*z

tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])

In [8]:
a

tensor([[[1.]],

        [[1.]]])

In [5]:
delta =0.1
loss_rate_of_change_w =(loss_fn(model(t_u, w+delta, b), t_c) -
                        loss_fn(model(t_u, w-delta, b), t_c)) /(2.0*delta)

learning_rate =1e-2  #o.o1
w = w-learning_rate * loss_rate_of_change_w

loss_rate_of_change_b =(loss_fn(model(t_u, w, b+delta), t_c) -
                        loss_fn(model(t_u, w, b-delta), t_c)) /(2.0*delta)
b = b-learning_rate * loss_rate_of_change_b

In [9]:
def loss_fn(t_p, t_c): #定義損失函數
    squared_diffs = (t_p - t_c)**2
    return squared_diffs.mean()

def dloss_fn(t_p, t_c):
    dsq_diffs = 2*(t_p - t_c)
    return dsq_diffs/ t_p.size(0) #由於要取平均，所以這裡需要進行除法
                      # t_p 張量內的元素總數

In [6]:
def model(t_u, w, b):
    return w * t_u +b

def dmodel_dw(t_u, w, b):
    return t_u

def dmodel_db(t_u, w, b):
    return 1.0

def grad_fn(t_u, t_c, t_p, w, b):
    dloss_dtp = dloss_fn(t_p, t_c)
    dloss_dw =  dloss_dtp * dmodel_dw(t_u, w, b)
    dloss_db = dloss_dtp * dmodel_db(t_u, w, b)
    return torch.stack([dloss_dw.sum(), dloss_db.sum()]) #將損失對w和b之導數堆疊在一起

In [7]:
def training_loop(n_epochs, learning_rate, params, t_u, t_c, print_params=True):
                  #訓練次數                  #包含參數w和b的tuple
    for epoch in range(1, n_epochs+1):
        w, b = params
        t_p = model(t_u, w, b)
        loss = loss_fn(t_p, t_c)
        grad = grad_fn(t_u, t_c, t_p, w, b) #計算梯度
        params = params - learning_rate* grad
        print('Epoch %d: Loss %f' %(epoch, float(loss)))
        if(print_params):
            print('\tParams: ', params)
            print("\tGrad: ", grad)
    return params

In [10]:
training_loop(n_epochs=100,
              learning_rate=1e-2,
              params=torch.tensor([1.0,0.0]),
              t_u=t_u,
              t_c=t_c)

Epoch 1: Loss 1763.884644
	Params:  tensor([-44.1730,  -0.8260])
	Grad:  tensor([4517.2964,   82.6000])
Epoch 2: Loss 5802484.500000
	Params:  tensor([2568.4011,   45.1637])
	Grad:  tensor([-261257.4062,   -4598.9707])
Epoch 3: Loss 19408033792.000000
	Params:  tensor([-148527.7344,   -2616.3931])
	Grad:  tensor([15109614.0000,   266155.6875])
Epoch 4: Loss 64915909902336.000000
	Params:  tensor([8589999.0000,  151310.8906])
	Grad:  tensor([-8.7385e+08, -1.5393e+07])
Epoch 5: Loss 217130559820791808.000000
	Params:  tensor([-4.9680e+08, -8.7510e+06])
	Grad:  tensor([5.0539e+10, 8.9023e+08])
Epoch 6: Loss 726257512784183951360.000000
	Params:  tensor([2.8732e+10, 5.0610e+08])
	Grad:  tensor([-2.9229e+12, -5.1486e+10])
Epoch 7: Loss 2429183416467662896627712.000000
	Params:  tensor([-1.6617e+12, -2.9270e+10])
	Grad:  tensor([1.6904e+14, 2.9776e+12])
Epoch 8: Loss 8125122549611731432050262016.000000
	Params:  tensor([9.6102e+13, 1.6928e+12])
	Grad:  tensor([-9.7764e+15, -1.7221e+14])
Epoc

tensor([nan, nan])

In [11]:
training_loop(n_epochs=100,
              learning_rate=1e-4,
              params=torch.tensor([1.0,0.0]),
              t_u=t_u,
              t_c=t_c)

Epoch 1: Loss 1763.884644
	Params:  tensor([ 0.5483, -0.0083])
	Grad:  tensor([4517.2964,   82.6000])
Epoch 2: Loss 323.090546
	Params:  tensor([ 0.3623, -0.0118])
	Grad:  tensor([1859.5492,   35.7843])
Epoch 3: Loss 78.929634
	Params:  tensor([ 0.2858, -0.0135])
	Grad:  tensor([765.4666,  16.5122])
Epoch 4: Loss 37.552845
	Params:  tensor([ 0.2543, -0.0143])
	Grad:  tensor([315.0790,   8.5787])
Epoch 5: Loss 30.540283
	Params:  tensor([ 0.2413, -0.0149])
	Grad:  tensor([129.6733,   5.3127])
Epoch 6: Loss 29.351158
	Params:  tensor([ 0.2360, -0.0153])
	Grad:  tensor([53.3495,  3.9682])
Epoch 7: Loss 29.148882
	Params:  tensor([ 0.2338, -0.0156])
	Grad:  tensor([21.9303,  3.4148])
Epoch 8: Loss 29.113848
	Params:  tensor([ 0.2329, -0.0159])
	Grad:  tensor([8.9964, 3.1869])
Epoch 9: Loss 29.107145
	Params:  tensor([ 0.2325, -0.0162])
	Grad:  tensor([3.6721, 3.0930])
Epoch 10: Loss 29.105247
	Params:  tensor([ 0.2324, -0.0166])
	Grad:  tensor([1.4803, 3.0544])
Epoch 11: Loss 29.104168
	Pa

tensor([ 0.2327, -0.0438])